In [1]:
import vertexai
from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search
from dotenv import load_dotenv
import os

load_dotenv()

PROJECT_ID = "adroit-sol-501711-r0"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://adroit-sol-501711-r0-adk-staging"

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

print("Vertex AI initialized.")

Vertex AI initialized.


In [ ]:
from callbacks import log_before_model, validate_input, log_after_model

GEMINI_MODEL = "gemini-2.5-flash"

STATE_SEARCH_RESULT = "search_result"
STATE_CRITIQUE = "critique"


search_agent = Agent(
    name="search_agent",
    model=GEMINI_MODEL,
    description="Searches the web to find information needed to answer the question.",
    instruction=(
        "Search the web for information that answers the user's question. "
        "Provide a clear, factual draft answer based on what you find."
    ),
    tools=[google_search],
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,
    output_key=STATE_SEARCH_RESULT,
    before_model_callback=log_before_model,
    after_model_callback=log_after_model,
)

critique_agent = Agent(
    name="critique_agent",
    model=GEMINI_MODEL,
    description="Reviews the draft answer and suggests specific improvements.",
    instruction=(
        f"Review this draft answer:\n\n{{{STATE_SEARCH_RESULT}}}\n\n"
        "Identify anything missing, unclear, inaccurate, or poorly explained. "
        "Provide specific, actionable suggestions for improvement. "
        "Do not rewrite the answer yourself — only critique it."
    ),
    output_key=STATE_CRITIQUE,
    before_model_callback=log_before_model,
    after_model_callback=log_after_model,
)

refine_agent = Agent(
    name="refine_agent",
    model=GEMINI_MODEL,
    description="Rewrites the draft answer based on the critique's suggestions.",
    instruction=(
        f"Original draft answer:\n{{{STATE_SEARCH_RESULT}}}\n\n"
        f"Critique and suggested improvements:\n{{{STATE_CRITIQUE}}}\n\n"
        "Rewrite the answer, applying the suggested improvements. "
        "Output ONLY the final, polished answer — no explanations or "
        "meta-commentary about the changes you made."
    ),
    before_model_callback=log_before_model,
    after_model_callback=log_after_model,
)

answer_team = SequentialAgent(
    name="answer_team",
    description="Answers a question by searching for data, critiquing the draft, and refining it.",
    sub_agents=[search_agent, critique_agent, refine_agent],
)

root_agent = Agent(
    name="root_agent",
    model=GEMINI_MODEL,
    description="Greets the user and routes real questions to the answer team.",
    instruction=(
        "You are a friendly greeter. If the user is just greeting you or "
        "making small talk, respond warmly and briefly yourself.\n\n"
        "If the user asks an actual question that needs a researched answer, "
        "delegate to answer_team — do not try to answer it yourself."
    ),
    sub_agents=[answer_team],
    before_model_callback=[log_before_model, validate_input],
    after_model_callback=log_after_model,
)

print("Agent defined:", root_agent.name)

In [ ]:
from vertexai.preview.reasoning_engines import AdkApp

app = AdkApp(agent=root_agent)

for event in app.stream_query(
    user_id="test_user",
    message="What are the health benefits of green tea?",
):
    print(event)

In [6]:
from vertexai import agent_engines

remote_agent = agent_engines.create(
    app,
    requirements=[
        "google-cloud-aiplatform[agent_engines,adk]",
    ],
    extra_packages=["callbacks.py"],
    env_vars={
        "GOOGLE_GENAI_USE_VERTEXAI": "true",
    },
)

print("Deployed agent resource name:", remote_agent.resource_name)

Identified the following requirements: {'pydantic': '2.13.4', 'google-cloud-aiplatform': '1.160.0', 'cloudpickle': '3.1.2'}
The following requirements are missing: {'pydantic', 'cloudpickle'}
The following requirements are appended: {'pydantic==2.13.4', 'cloudpickle==3.1.2'}
The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'pydantic==2.13.4', 'cloudpickle==3.1.2']
Using bucket adroit-sol-501711-r0-adk-staging
Wrote to gs://adroit-sol-501711-r0-adk-staging/agent_engine/agent_engine.pkl
Writing to gs://adroit-sol-501711-r0-adk-staging/agent_engine/requirements.txt
Creating in-memory tarfile of extra_packages
Writing to gs://adroit-sol-501711-r0-adk-staging/agent_engine/dependencies.tar.gz
Bidi stream API mode is not supported yet in Vertex SDK, please use the GenAI SDK instead. Skipping method bidi_stream_query.
Creating AgentEngine
Create AgentEngine backing LRO: projects/735570248844/locations/us-central1/reasoningEngines/3994796773326454784/operations/531

In [8]:
for event in remote_agent.stream_query(
    user_id="deployment_test_user",
    message="Hi there!",
):
    print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': 'Hello! How can I help you today?'}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 9, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 9}], 'prompt_token_count': 285, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 285}], 'total_token_count': 294, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -0.2649389108022054, 'invocation_id': 'e-e435e30d-1f13-4114-a7e0-1b7b75841f2f', 'author': 'root_agent', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'node_info': {'path': 'root_agent@1'}, 'id': 'c3c34221-0364-45ac-bb42-af1855f45609', 'timestamp': 1783613355.0885947}


In [7]:
for event in remote_agent.stream_query(
    user_id="deployment_test_user",
    message="What are the health benefits of green tea?",
):
    print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-990dd11e-f723-4837-83c4-008009b871a3', 'args': {'agent_name': 'answer_team'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'CqEEAY89a1-37dH37qEXUJrylVHvtwlOsxsgDhWbu47K0BMC5NttSfENsjvhBKpbyHqej4MEBpRoJ8usNE5RBmS-vtGg0n4N0pb0LRfURynG2oydQryHfO9ipbd7w5qHHQviXenH-EQDuEFEKKIeLT-ggSPtHj9QkIFfeuRMc4YbAKxh-Vs9VDZSpLxOGFBQKEC3CHz10B2CrG2wKWTVjngwq3KUBe88HNh6s1quxv6asBQp05UqiJGNaD8ycA2AAdpZBD-lZM2rkhwyN5roJFaV84K-8LOTCs71Qkt9DDNzUaNi41r9j4dXZv3BKih6CaouBAs1q7Gz65MhBQs-lbbLGoNKq2pBmVASNFrE93DPROe8-a7e2nMbD8KDFUvVg4BQjP2hxSII5h-N9NMknBVcE0SiXyVYR03up4MBSmz1LXYTkDIficR3mQfj5HQa1yTF_GU6Xw5cfErdmjAXvSqsXCtqZgKOLr837juImNUthg46iNh3kmhuZagQi9SmaZaNYADtwz5I8VqRiK82lyoio-6RV4emrfWsiO-wQ009S80RZ-mm0OSSNqX1Lxs5UKMNsJL9ZU7PDopmt8t2NTk-q0_OkDv1_bAjrw5NdXG-GH3ooj1OlNE-qHV8XhAKWWKLbQy64eFLB4NpRzz_ARYCg2VdZt0_EcR3h7T42-p6G9q4_gedPIcrWNt0Wb1a0zmTXnovnqvDFA66XqbpHd4dOz8='}], 'role': 'model'}, 'finish_reason': 'STOP', 